# KNPS 2026 해안쓰레기 — 새 베이스라인 추론 (config 구동)

- 입력: `AIF_INPUT_DIR/patches.csv`(`id`) + `images/<id>.png` (256x256 RGB 8bit)
- 출력: `AIF_PREDICTION_PATH` 에 `id,rle` CSV (행 우선, 0-based `시작 길이` 쌍, 빈 문자열 = 쓰레기 없음)
- 모델·임계값·TTA 는 `assets/config.json`, 가중치는 `assets/model/*.pt` (네트워크 불필요)

In [ ]:
# [setup]
import csv, io, json, os, sys, time, traceback
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

T_START = time.time()
INPUT_DIR = Path(os.environ.get("AIF_INPUT_DIR", "./input"))
PREDICTION_PATH = Path(os.environ.get("AIF_PREDICTION_PATH", "./prediction.csv"))
ASSETS = Path("assets")
CFG = json.loads((ASSETS / "config.json").read_text(encoding="utf-8"))
H = W = 256
MIN_POS_PX = 50                                   # 채점 규칙: 1인 화소 50개 미만 = 빈 예측
MEAN = np.array(CFG.get("mean", [0.485, 0.456, 0.406]), np.float32)
STD = np.array(CFG.get("std", [0.229, 0.224, 0.225]), np.float32)
TIME_BUDGET_S = float(CFG.get("time_budget_s", 95 * 60))   # 120분 한도에서 설치·채점 몫을 뺀 추론 예산
print(json.dumps({k: v for k, v in CFG.items() if k != "notes"}, ensure_ascii=False))


def rle_encode(mask):
    m = np.asarray(mask, dtype=bool).reshape(-1)
    d = np.diff(np.concatenate([[0], m.astype(np.int8), [0]]))
    st, en = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    return " ".join(f"{a} {b - a}" for a, b in zip(st.tolist(), en.tolist()))


def rle_check(s):
    """주최 측 거부 규칙(정수 아님/홀수 개/길이<1/범위 밖)을 저장 전에 스스로 검사한다. 통과 시 화소 수 반환."""
    if not s:
        return 0
    v = [int(t) for t in s.split()]
    assert len(v) % 2 == 0
    n = 0
    for a, l in zip(v[0::2], v[1::2]):
        assert l >= 1 and 0 <= a and a + l <= H * W
        n += l
    return n

In [ ]:
# [inputs] 목록 파일이 정본. 껍데기 폴더 한 겹/압축본 그대로 주어지는 경우까지 대비(베이스라인과 동일 규약)
print("AIF_INPUT_DIR:", INPUT_DIR, sorted(p.name for p in INPUT_DIR.iterdir())[:10] if INPUT_DIR.is_dir() else "(없음)")
lists = sorted(INPUT_DIR.rglob("patches.csv"))
if not lists:
    zs = sorted(INPUT_DIR.rglob("*.zip")) if INPUT_DIR.is_dir() else []
    if len(zs) == 1:
        import tempfile, zipfile
        dst = Path(tempfile.mkdtemp()) / "input"
        with zipfile.ZipFile(zs[0]) as z:
            z.extractall(dst)
        print(f"입력이 압축본이어서 풀어서 읽습니다: {zs[0].name} -> {dst}")
        lists = sorted(dst.rglob("patches.csv"))
if not lists:
    raise SystemExit(f"patches.csv 를 찾지 못했습니다: {INPUT_DIR}")
ROOT = lists[0].parent
with lists[0].open(encoding="utf-8-sig", newline="") as f:
    IDS = [row["id"].strip() for row in csv.DictReader(f) if row.get("id", "").strip()]
IDS = list(dict.fromkeys(IDS))                     # 중복 id 는 제출 전체 실패 사유 → 순서 보존 중복 제거
print(f"입력 루트 {ROOT}, 패치 {len(IDS)}건")

In [ ]:
# [vfm] vfm_net.py 본문 — v22 마스크 모델(Built with DINOv3; AnyUp CC BY 4.0, assets/code/anyup)
import sys as _sys
_sys.path.insert(0, str(ASSETS / "code"))
"""vfm_net.py — 얼린 비전 기반모델(DINOv3 위성판 / DINOv2) + Rein 방식 경량 어댑터 + 고해상 상세 가지 디코더.

합성→실제 일반화 분할 연구(Rein CVPR 2024, Rein++ 2025, DGSS 서베이 2025)의 공통 결론: 기반모델 백본은 얼리고
층마다 작은 어댑터만 학습하면(백본의 1% 남짓) 사전학습이 가진 실영상 지식이 합성 데이터에 덮어써지지 않아
GTA→실도시 mIoU 가 ResNet 계열 48% → 68% 로 오른다. 우리 문제(합성 학습 → 실제 위성 평가)와 같은 구조다.

- 백본: timm 이름. 'vit_large_patch16_dinov3.sat493m'(Maxar 0.6 m 위성 4.9억 장, DINOv3 라이선스: 상업·파생 허용, "Built with DINOv3" 표기)
        또는 'vit_large_patch14_reg4_dinov2.lvd142m'(Apache-2.0).
- 어댑터(Rein 간소판): 블록마다 학습 토큰 100개(저랭크) — 특징과 토큰의 유사도로 잔차 보정, 게이트 0 에서 시작(처음엔 원래 특징 그대로).
- 디코더: 4개 층 특징으로 피라미드(stride 4·8·16·32) → 위에서 아래로 합치며, 원해상도 상세 CNN 가지(stride 1·2·4)를 붙여 1 px 경계를 낸다.
- train.Net 이 arch="VFM" 일 때 body 로 쓴다. 입력은 Net 이 upscale 배 키운 정규화 영상(ImageNet 평균·표준편차), 출력은 (같은 해상도 마스크 logit, 존재 logit).
"""
import math

import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

IMN_MEAN = (0.485, 0.456, 0.406)
IMN_STD = (0.229, 0.224, 0.225)
ANYUP_MB = 8                      # AnyUp 한 번에 처리하는 장 수
ANYUP_CHUNK = 4096                # AnyUp 쿼리(출력 화소)를 이만큼씩 나눠 주의를 계산 — 메모리 절약(결과는 같음)
TOKENS = 32                       # 백본 입력을 32x32 토큰 격자로 맞춘다(p16 → 512 px, p14 → 448 px)


class Rein(nn.Module):
    def __init__(self, dim, m=100, r=16, v=64, pos_r=0):
        super().__init__()
        self.a = nn.Parameter(torch.randn(m, r) * 0.02)
        self.b = nn.Parameter(torch.randn(r, dim) * 0.02)
        self.v1 = nn.Linear(dim, v)
        self.v2 = nn.Linear(v, dim)
        self.gate = nn.Parameter(torch.zeros(1))
        if pos_r:                                                    # v22: Rein 원본의 (Δf̄ + f)·W_f 중 특징 자체의 선형 변환 f·W_f (저랭크 pos_r)
            self.pu = nn.Parameter(torch.randn(dim, pos_r) / math.sqrt(dim))   # 위치(토큰)마다 다른 보정을 만든다 — v19 간소판에는 없던 부분
            self.pv = nn.Parameter(torch.zeros(pos_r, dim))          # 0 초기화: 처음엔 v19 와 출력이 같다

    def forward(self, f):                                            # f: (B, N, C) 블록 출력(접두 토큰 포함)
        t = self.a @ self.b                                          # (m, C)
        s = torch.softmax((f @ t.t().to(f.dtype)) / math.sqrt(f.shape[-1]), dim=-1)
        out = f + self.gate.to(f.dtype) * (s @ self.v2(F.gelu(self.v1(t))).to(f.dtype))
        if hasattr(self, "pv"):
            out = out + (f @ self.pu.to(f.dtype)) @ self.pv.to(f.dtype)
        return out


def _rein_hook(mod, inp, out):                                       # 모듈 함수 + mod.rein → deepcopy(EMA) 해도 사본은 자기 어댑터를 쓴다
    return mod.rein(out)


def cbr(i, o, s=1):
    return nn.Sequential(nn.Conv2d(i, o, 3, s, 1, bias=False), nn.BatchNorm2d(o), nn.GELU())


class VFMBody(nn.Module):
    def __init__(self, name, pretrained=True, dec=256, tokens=TOKENS, rein_pos=0, anyup=False, anyup_ckpt=None):
        super().__init__()
        if tokens != TOKENS:                                          # v21: 격자를 바꾼 모델은 표식 버퍼를 둔다 → 다른 격자로 만든 모델에
            self.register_buffer("grid_tokens", torch.tensor(int(tokens)))   # strict 로드하면 오류가 나서 조용히 틀리지 않는다
        self.vfm = timm.create_model(name, pretrained=pretrained, num_classes=0, dynamic_img_size=True)
        for p in self.vfm.parameters():
            p.requires_grad_(False)
        cfg = self.vfm.pretrained_cfg
        ps = self.vfm.patch_embed.patch_size
        self.res = int(tokens) * (ps[0] if isinstance(ps, (tuple, list)) else ps)
        C, L = self.vfm.embed_dim, len(self.vfm.blocks)
        self.idx = [L // 4 - 1, L // 2 - 1, 3 * L // 4 - 1, L - 1]
        for blk in self.vfm.blocks:
            blk.rein = Rein(C, pos_r=rein_pos)
            blk.register_forward_hook(_rein_hook)
        m = torch.tensor(cfg.get("mean", IMN_MEAN)).view(1, 3, 1, 1)
        s = torch.tensor(cfg.get("std", IMN_STD)).view(1, 3, 1, 1)
        self.register_buffer("in_a", torch.tensor(IMN_STD).view(1, 3, 1, 1) / s, persistent=False)
        self.register_buffer("in_b", (torch.tensor(IMN_MEAN).view(1, 3, 1, 1) - m) / s, persistent=False)
        self.lat = nn.ModuleList([nn.Sequential(nn.Conv2d(C, dec, 1, bias=False), nn.GroupNorm(32, dec)) for _ in range(4)])
        self.td = nn.ModuleList([cbr(dec, dec) for _ in range(3)])        # s32→s16, s16→s8, s8→s4
        self.d1, self.d2, self.d4 = cbr(3, 32), cbr(32, 64, 2), cbr(64, 96, 2)
        self.f4 = cbr(dec + 96, 128)
        self.f2 = cbr(128 + 64, 64)
        self.f1 = cbr(64 + 32, 32)
        self.out = nn.Conv2d(32, 1, 1)
        if anyup:                                                    # v22: AnyUp(ICLR 2026, CC BY 4.0) — 영상 밝기 경계를 길잡이로 DINO 특징을 원해상도로 복원
            from anyup.model import AnyUp
            self.anyup = AnyUp()
            if anyup_ckpt:
                self.anyup.load_state_dict(torch.load(anyup_ckpt, map_location="cpu", weights_only=True))
            for p in self.anyup.parameters():
                p.requires_grad_(False)                              # 사전학습 업샘플러는 얼림(인코더와 무관하게 일반화하는 성질 유지)
            self.any_fuse = nn.Conv2d(dec, 128, 1)
            nn.init.zeros_(self.any_fuse.weight); nn.init.zeros_(self.any_fuse.bias)   # 0 초기화: 처음엔 v19 와 출력이 같다
        self.cls = nn.Sequential(nn.Linear(C, 256), nn.GELU(), nn.Dropout(0.2), nn.Linear(256, 1))

    def forward(self, x):                                            # x: (B,3,H,W) ImageNet 정규화, H=W=256*upscale
        H, W = x.shape[-2:]
        v = F.interpolate(x, size=(self.res, self.res), mode="bilinear", align_corners=False) * self.in_a + self.in_b
        feats = self.vfm.forward_intermediates(v, indices=self.idx, norm=False, output_fmt="NCHW", intermediates_only=True)
        p4 = F.interpolate(self.lat[0](feats[0]), scale_factor=4, mode="bilinear", align_corners=False)
        p8 = F.interpolate(self.lat[1](feats[1]), scale_factor=2, mode="bilinear", align_corners=False)
        p16 = self.lat[2](feats[2])
        p32 = F.avg_pool2d(self.lat[3](feats[3]), 2)
        y = self.td[0](F.interpolate(p32, scale_factor=2, mode="bilinear", align_corners=False) + p16)
        y = self.td[1](F.interpolate(y, scale_factor=2, mode="bilinear", align_corners=False) + p8)
        y = self.td[2](F.interpolate(y, scale_factor=2, mode="bilinear", align_corners=False) + p4)   # stride 4 of 512 = 128
        e1 = self.d1(x); e2 = self.d2(e1); e4 = self.d4(e2)
        y = self.f4(torch.cat([F.interpolate(y, size=e4.shape[-2:], mode="bilinear", align_corners=False), e4], 1))
        y = F.interpolate(y, size=e2.shape[-2:], mode="bilinear", align_corners=False)
        if hasattr(self, "anyup"):                                   # 원해상도(256) 특징을 AnyUp 으로 만들어 stride 2 단계에 더한다
            with torch.no_grad():
                img = F.avg_pool2d(x.float(), x.shape[-1] // e2.shape[-1]) if x.shape[-1] != e2.shape[-1] else x.float()
                lf = self.lat[3](feats[3]).float()
                nk = lf.shape[-2] * lf.shape[-1]                       # AnyUp 은 8장씩(내부 합성곱·주의 텐서의 32비트 색인 한계), 쿼리도 나눠 계산 — 결과는 같다
                chunk = max(256, min(ANYUP_CHUNK, (1 << 29) // (ANYUP_MB * 4 * nk)))
                a_hr = torch.cat([self.anyup(img[i:i + ANYUP_MB], lf[i:i + ANYUP_MB], output_size=tuple(e2.shape[-2:]), q_chunk_size=chunk)
                                  for i in range(0, img.shape[0], ANYUP_MB)])
            y = y + self.any_fuse(a_hr.to(y.dtype))
        y = self.f2(torch.cat([y, e2], 1))
        y = self.f1(torch.cat([F.interpolate(y, size=e1.shape[-2:], mode="bilinear", align_corners=False), e1], 1))
        logit = self.out(y)
        if logit.shape[-2:] != (H, W):
            logit = F.interpolate(logit, size=(H, W), mode="bilinear", align_corners=False)
        cls = self.cls(feats[3].float().mean((2, 3)))
        return logit, cls


In [ ]:
# [model] self-contained standard/detail models and independent head weights
import math
import segmentation_models_pytorch as smp
print("python", sys.version.split()[0], "torch", torch.__version__, "smp", smp.__version__, "cuda", torch.cuda.is_available())


class BaseLogits(torch.nn.Module):
    def __init__(self, cfg):
        super().__init__()
        aux = dict(classes=1, dropout=0.0) if cfg.get("aux_presence", True) else None
        if cfg["arch"] == "VFM":                           # v19: 얼린 기반모델 + Rein 어댑터 + 고해상 디코더 (vfm_net.py 셀)
            self.body = VFMBody(cfg["encoder"], pretrained=False, tokens=int(cfg.get("vfm_tokens", 32)),
                                rein_pos=int(cfg.get("vfm_rein_pos", 0)), anyup=bool(cfg.get("vfm_anyup", False)))
        else:
            self.body = getattr(smp, cfg["arch"])(encoder_name=cfg["encoder"], encoder_weights=None,
                                                 in_channels=3, classes=1, aux_params=aux)
        self.up = int(cfg.get("upscale", 2))

    def forward(self, x):
        if self.up > 1:
            x = F.interpolate(x, scale_factor=self.up, mode="bilinear", align_corners=False)
        out = self.body(x)
        return out if isinstance(out, tuple) else (out, None)


def output_probabilities(logit, cls, up):
    prob = torch.sigmoid(logit.float())
    if up > 1:
        prob = F.avg_pool2d(prob, up)
    pres = torch.sigmoid(cls.float()).flatten() if cls is not None else prob.flatten(1).amax(1)
    return prob[:, 0], pres


class Net(BaseLogits):
    def forward(self, x):
        logits, presence = super().forward(x)
        return output_probabilities(logits, presence, self.up)


class DetailNet(torch.nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.base = BaseLogits(cfg)
        self.up = self.base.up
        detail = cfg.get("detail_config", {})
        channels = int(detail.get("channels", 24))
        groups = next(g for g in (8, 4, 2, 1) if channels % g == 0)
        self.detail = torch.nn.Sequential(
            torch.nn.Conv2d(3, channels, 3, padding=1, bias=False),
            torch.nn.GroupNorm(groups, channels),
            torch.nn.SiLU(inplace=True),
            torch.nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            torch.nn.GroupNorm(groups, channels),
            torch.nn.SiLU(inplace=True),
        )
        self.correction = torch.nn.Conv2d(channels, 1, 1)
        self.density_head = torch.nn.Conv2d(channels, 1, 1) if detail.get("auxiliary_density", False) else None

    def forward(self, x):
        logits, presence = self.base(x)
        correction = self.correction(self.detail(x))
        if correction.shape[-2:] != logits.shape[-2:]:
            correction = F.interpolate(correction, size=logits.shape[-2:], mode="bilinear", align_corners=False)
        return output_probabilities(logits + correction, presence, self.up)


def load_models(device):
    nets = []
    for spec in CFG["members"]:
        wm, wc = float(spec["mask_weight"]), float(spec["presence_weight"])
        if not all(math.isfinite(v) and v >= 0 for v in (wm, wc)):
            raise ValueError("Head weights must be finite and nonnegative")
        if wm == wc == 0 and not float(spec.get("rescue_weight", 0)):
            raise ValueError("An unused model member is not allowed")
        variant = spec.get("model_variant", "standard")
        if variant not in ("standard", "detail_residual"):
            raise ValueError("Unknown model variant: " + variant)
        if not spec["weights"]:
            raise ValueError("Missing configured weights")
        for wname in spec["weights"]:
            net = DetailNet(spec) if variant == "detail_residual" else Net(spec)
            state = torch.load(ASSETS / "model" / wname, map_location="cpu", weights_only=True)
            net.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in state.items()}, strict=True)
            n = len(spec["weights"])
            net.rw = float(spec.get("rescue_weight", 0)) / n      # v22: 구제 판단 전용 멤버 가중치
            nets.append((net.to(device).eval(), wm / n, wc / n))
    if sum(wm for _, wm, _ in nets) <= 0 or sum(wc for _, _, wc in nets) <= 0:
        raise ValueError("Both heads must have positive total weight")
    return nets


def tta_views(mode):
    full = [(k, f) for f in (False, True) for k in range(4)]
    return {"none": full[:1], "flip": [full[0], full[4]], "d4": full}[mode]


@torch.no_grad()
def predict_batch(nets, imgs, device, tta, allow_fallback=False):
    """v22 강건성: 판정 경로(존재 확률·v4 자체 마스크)는 항상 D4 8회(gate_tta) — 시간 가드가 판정을 바꾸지 않는다.
    마스크 경로만 tta(시간 가드가 줄일 수 있음). allow_fallback 이면 마스크 모델 오류 시 v4 자체 마스크로 대체한다."""
    x = np.stack([(im / 255.0 - MEAN) / STD for im in imgs]).astype(np.float32)
    x = torch.from_numpy(x).permute(0, 3, 1, 2).to(device)
    use_amp = device == "cuda" and CFG.get("fp16", True)

    def run(views, pick):
        acc_m, acc_c, n = 0.0, 0.0, 0.0
        for k, flip in views:
            xt = torch.rot90(x.flip(-1) if flip else x, k, (-2, -1))
            for net, wm, wc in nets:
                w = pick(wm, wc)
                if not w:
                    continue
                with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                    p, c = net(xt)
                p = torch.rot90(p, -k, (-2, -1))
                acc_m = acc_m + w * (p.flip(-1) if flip else p)
                acc_c = acc_c + w * c
                n += w
        return acc_m, acc_c, n

    acc_q, acc_c, n_c = run(tta_views(CFG.get("gate_tta", "d4")), lambda wm, wc: wc)
    if n_c <= 0:
        raise ValueError("Presence head must have positive total weight")
    q = (acc_q / n_c).float().cpu().numpy()
    pres = (acc_c / n_c).float().cpu().numpy()
    rprob, rs = None, CFG.get("rescue")
    rnets = [(net, getattr(net, "rw", 0.0)) for net, _, _ in nets if getattr(net, "rw", 0.0) > 0]
    if rs and rnets:                                   # 구제 판단 전용 확률(Public 에서 검증된 v19_d3): 구제 후보 구간 패치만, 판정이므로 D4 고정
        band = np.flatnonzero((pres >= float(rs["lo"])) & (pres < float(CFG["presence_thr"])))
        rprob = np.zeros_like(q)
        if len(band):
            try:
                xb, acc_r, n_r = x[torch.as_tensor(band, device=x.device)], 0.0, 0.0
                for k, flip in tta_views(CFG.get("gate_tta", "d4")):
                    xt = torch.rot90(xb.flip(-1) if flip else xb, k, (-2, -1))
                    for net, rw in rnets:
                        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                            p, _ = net(xt)
                        p = torch.rot90(p, -k, (-2, -1))
                        acc_r = acc_r + rw * (p.flip(-1) if flip else p)
                        n_r += rw
                rprob[band] = (acc_r / n_r).float().cpu().numpy()
            except Exception:  # noqa: BLE001
                if not allow_fallback:
                    raise
                traceback.print_exc()
                rprob = None                               # 실패하면 구제 판단은 마스크 확률로
    try:
        if CFG.get("_test_fail_mask"):
            raise RuntimeError("test: simulated mask-model failure")
        acc_p, _, n_p = run(tta_views(tta), lambda wm, wc: wm)
        if n_p <= 0:
            raise ValueError("Mask head must have positive total weight")
        prob = (acc_p / n_p).float().cpu().numpy()
    except Exception:  # noqa: BLE001
        if not allow_fallback:
            raise
        traceback.print_exc()
        if device == "cuda":
            torch.cuda.empty_cache()
        st = globals().get("STATE")
        if st is not None:
            st["mask_fallback"] = st.get("mask_fallback", 0) + len(imgs)
        prob = q                                       # 판정은 그대로, 마스크만 v4 자체 마스크로
    return prob, pres, q, rprob


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
try:
    NETS = load_models(DEVICE)
    predict_batch(NETS, [np.zeros((H, W, 3), np.uint8)], DEVICE, "none")
except Exception as e:
    print(f"{DEVICE} 시험 추론 실패({type(e).__name__}: {e}) → CPU 전환")
    DEVICE = "cpu"
    NETS = load_models(DEVICE)
if DEVICE == "cpu":
    torch.set_num_threads(max(1, (os.cpu_count() or 4)))
print("device", DEVICE, "models", len(NETS), "head_weights", [(wm, wc) for _, wm, wc in NETS])


In [ ]:
# [verifier] v16 2차 분류기: DINOv2-B reg4(Apache-2.0) 동결 특징 + MLP 헤드.
# 1차(v4 존재)가 양성이라 한 패치만 다시 보고 p(쓰레기) < tau 면 음성(빈 마스크)으로 돌린다. 마스크는 그대로.
import timm
VCFG = CFG.get("verifier") or {}
VERIFIER = None
_IMN_M = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
_IMN_S = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class _VHead(torch.nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(d, 512), torch.nn.GELU(), torch.nn.Dropout(0.2), torch.nn.Linear(512, 128),
                                       torch.nn.GELU(), torch.nn.Dropout(0.1), torch.nn.Linear(128, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)


def _v_scalars(prob, pres):
    prob = np.asarray(prob, np.float32); pres = np.asarray(pres, np.float32)
    area = (prob >= 0.4).reshape(len(prob), -1).sum(1)
    pm = np.where(prob >= 0.4, prob, 0).reshape(len(prob), -1).sum(1) / np.maximum(area, 1)
    pc = np.clip(pres, 1e-4, 1 - 1e-4)
    return np.stack([np.log(pc / (1 - pc)), np.log1p(area), prob.reshape(len(prob), -1).max(1), pm], 1).astype(np.float32)


if VCFG.get("enabled", False):
    try:
        _dino = timm.create_model("vit_base_patch14_reg4_dinov2", pretrained=False, num_classes=0, img_size=448)
        _sd = torch.load(ASSETS / "model" / VCFG["backbone"], map_location="cpu", weights_only=True)
        _dino.load_state_dict({k: v.float() for k, v in _sd.items()})
        _dino = _dino.to(DEVICE).eval()
        _hd = torch.load(ASSETS / "model" / VCFG["head"], map_location="cpu", weights_only=True)
        _vhead = _VHead(int(_hd["d"]))
        _vhead.load_state_dict(_hd["state"])
        _vhead = _vhead.to(DEVICE).eval()
        _vmu, _vsd = _hd["mu"].float().to(DEVICE), _hd["sd"].float().to(DEVICE)
        VERIFIER = dict(tau=float(VCFG["tau"]))
        print("verifier 로드: tau", VERIFIER["tau"])
    except Exception as e:  # noqa: BLE001
        print(f"verifier 로드 실패({type(e).__name__}: {e}) → 1차 판정만 사용")
        VERIFIER = None


@torch.no_grad()
def verify_batch(imgs, qprob, pres, bs=32):
    out = []
    for s in range(0, len(imgs), bs):
        x = torch.from_numpy(np.stack(imgs[s:s + bs])).to(DEVICE).permute(0, 3, 1, 2).float() / 255.0
        x = (x - _IMN_M.to(DEVICE)) / _IMN_S.to(DEVICE)
        x = F.interpolate(x, size=448, mode="bilinear", align_corners=False)
        q = torch.from_numpy(np.asarray(qprob[s:s + bs], np.float32)).to(DEVICE)[:, None]
        acc = 0.0
        for flip in (False, True):
            xx, qq = (x.flip(-1), q.flip(-1)) if flip else (x, q)
            with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
                t = _dino.forward_features(xx)
            t = t.float()
            cls, pt = t[:, 0], t[:, _dino.num_prefix_tokens:]
            w = F.adaptive_avg_pool2d(qq, 32).flatten(1)
            w = w / (w.sum(1, keepdim=True) + 1e-6)
            acc = acc + torch.cat([cls, pt.mean(1), (pt * w[..., None]).sum(1)], 1)
        f = (acc / 2).half().float()                                     # 학습 특징(float16 저장)과 같은 반올림
        sc = torch.from_numpy(_v_scalars(qprob[s:s + bs], pres[s:s + bs])).to(DEVICE)
        z = (torch.cat([f, sc], 1) - _vmu) / _vsd
        out.append(torch.sigmoid(_vhead(z)).float().cpu().numpy())
    return np.concatenate(out)


In [ ]:
# [postprocess] postproc.py 본문(학습 때 임계 탐색에 쓴 것과 동일한 코드)
"""postproc.py — 지표 구조를 반영한 결정 규칙(순수 numpy). train.py 의 임계 탐색과 predict.ipynb 가 '같은 코드'를 쓴다
(make_submission.py 가 이 파일 본문을 노트북 셀에 그대로 심는다 → 학습 때 고른 임계가 추론 때와 어긋나지 않는다).

  * 존재 판정: FN 은 F1 과 그 패치의 형상 점수를 모두 잃고, FP 는 F1 만 잃는다 → presence_thr 는 검증셋에서 지표로 직접 고른다
  * 양성으로 판정했는데 마스크가 50 화소 미만이면 버리지 않고 50 화소 이상으로 부풀린다(버리면 그 패치는 0점 확정).
    가장 가까운 배경 화소부터 채우는데, GT 바깥 1 화소 고리는 정밀도 계산에서 공짜다(P 는 dilate1(gt) 안의 예측을 센다)
  * 음성으로 판정하면 완전한 빈 마스크(1~49 화소 찌꺼기는 어차피 빈 예측 취급)
"""
import numpy as np

MIN_POS_PX = 50


def dilate1(m):
    h, w = m.shape
    p = np.pad(m, 1)
    out = np.zeros_like(m)
    for dy in (0, 1, 2):
        for dx in (0, 1, 2):
            out |= p[dy:dy + h, dx:dx + w]
    return out


def inflate_to_min_area(mask, prob, target=56):
    m = mask.copy()
    if not m.any():                                    # 씨앗이 없으면 확률 최댓값 화소에서 시작
        m[np.unravel_index(int(np.argmax(prob)), prob.shape)] = True
    while int(m.sum()) < target:
        ring = dilate1(m) & ~m
        need = target - int(m.sum())
        if int(ring.sum()) <= need:
            m |= ring
        else:                                          # 마지막 고리는 확률이 높은 화소부터 필요한 만큼만
            ys, xs = np.nonzero(ring)
            keep = np.argsort(-prob[ys, xs], kind="stable")[:need]
            m[ys[keep], xs[keep]] = True
    return m


def decide(prob, pres, cfg, rprob=None):
    """prob: (H,W) 화소 확률, pres: 패치 존재 확률 → 제출용 bool 마스크."""
    mask = prob >= float(cfg["mask_thr"])
    area = int(mask.sum())
    positive = float(pres) >= float(cfg["presence_thr"]) or area >= int(cfg.get("area_strong", 10 ** 9))
    rs = cfg.get("rescue")                       # v21: 존재가 조금 낮아도 DINOv3 마스크가 강하게 확신하면 양성(SIIM 1위식 마스크 확신 면적 판정)
    if not positive and rs and float(pres) >= float(rs["lo"]) and int(((prob if rprob is None else rprob) >= float(rs["t"])).sum()) >= int(rs["n"]):
        positive = True
    if not positive:
        return np.zeros_like(mask)
    if area < MIN_POS_PX:
        mask = inflate_to_min_area(mask, prob, int(cfg.get("inflate_to", 56)))
    return mask

In [ ]:
# [chroma] chroma.py 본문 — config 의 input_chroma_sigma > 0 이면 학습 때와 같은 입력 색 정규화(v4 M6)
"""chroma.py — 입력 색 정규화 (v4 M6, 선택 기능. config 의 input_chroma_sigma > 0 일 때만 동작)

학습과 추론이 '같은 코드'로 입력 영상의 색(Cb/Cr)만 σ 만큼 번지게 한다. 휘도는 건드리지 않는다.
평가 영상의 색 해상도(팬샤픈 1.2 m 다중분광 색, JPEG 4:2:0 등)가 무엇이든 모델이 늘 같은 정도로 번진 색을 보게 하려는 장치.
순수 numpy — 채점 서버에는 opencv·scipy 가 없다. make_submission.py 가 이 파일 본문을 노트북 셀에 그대로 심는다.
"""
import numpy as np

_Y = np.array([0.299, 0.587, 0.114], np.float32)                  # BT.601 전범위(JPEG) 휘도


def _gauss_kernel(sigma):
    r = max(1, int(np.ceil(3.0 * sigma)))
    x = np.arange(-r, r + 1, dtype=np.float32)
    k = np.exp(-0.5 * (x / sigma) ** 2)
    return k / k.sum(), r


def _blur_hw(a, sigma):
    """a: (H,W,C) float32 → 가우시안 블러(분리형, 반사 패딩)."""
    k, r = _gauss_kernel(sigma)
    h, w = a.shape[:2]
    p = np.pad(a, ((r, r), (0, 0), (0, 0)), mode="reflect")
    a = sum(k[i] * p[i:i + h] for i in range(2 * r + 1))
    p = np.pad(a, ((0, 0), (r, r), (0, 0)), mode="reflect")
    return sum(k[i] * p[:, i:i + w] for i in range(2 * r + 1))


def canon_chroma(img, sigma):
    """img: (H,W,3) uint8 RGB → 색(Cb/Cr)만 σ 화소 번진 uint8 RGB. σ<=0 이면 그대로."""
    if not sigma or float(sigma) <= 0:
        return img
    f = img.astype(np.float32)
    y = f @ _Y
    c = np.stack([(f[..., 2] - y) * 0.564, (f[..., 0] - y) * 0.713], -1)   # Cb, Cr
    c = _blur_hw(c, float(sigma))
    r = y + 1.403 * c[..., 1]
    b = y + 1.773 * c[..., 0]
    g = (y - 0.299 * r - 0.114 * b) / 0.587
    return np.clip(np.stack([r, g, b], -1) + 0.5, 0, 255).astype(np.uint8)

In [ ]:
# [infer-def]
CHROMA_SIGMA = float(CFG.get("input_chroma_sigma", 0) or 0)
print("input_chroma_sigma", CHROMA_SIGMA)


def read_image(path):
    with Image.open(io.BytesIO(path.read_bytes())) as im:
        arr = np.asarray(im.convert("RGB"))
    if arr.shape[:2] != (H, W):                        # 규격 밖 입력도 죽지 않고 맞춰서 처리
        arr = np.asarray(Image.fromarray(arr).resize((W, H), Image.BILINEAR))
    return canon_chroma(arr, CHROMA_SIGMA) if CHROMA_SIGMA > 0 else arr


ROWS, STATE = {}, {"tta": CFG.get("tta", "d4"), "done": 0, "fail": 0, "t_inf": 0.0, "vrej": 0}
BATCH = int(CFG.get("batch", 32))
SHARDS = [IDS[i::1] for i in [0]] if len(IDS) < 64 else [IDS[(len(IDS) * k) // 4:(len(IDS) * (k + 1)) // 4] for k in range(4)]


def run_shard(k):
    if k >= len(SHARDS):
        return
    ids_all, t0 = SHARDS[k], time.time()
    for s in range(0, len(ids_all), BATCH):
        ids = ids_all[s:s + BATCH]
        imgs, ok = [], []
        for i in ids:
            try:
                imgs.append(read_image(ROOT / "images" / f"{i}.png")); ok.append(i)
            except Exception as e:  # noqa: BLE001
                ROWS[i] = ""; STATE["fail"] += 1
                print(f"  [skip] {i}: {type(e).__name__}: {e}")
        if ok:
            try:
                probs, press, qprobs, rprobs = predict_batch(NETS, imgs, DEVICE, STATE["tta"], allow_fallback=True)
                masks = [decide(p, c, CFG, None if rprobs is None else rprobs[j]) for j, (p, c) in enumerate(zip(probs, press))]
                pos_idx = [j for j, m in enumerate(masks) if int(m.sum()) >= MIN_POS_PX]
                if VERIFIER is not None and pos_idx:
                    try:
                        pv = verify_batch([imgs[j] for j in pos_idx], qprobs[pos_idx], press[pos_idx])
                        for j, v in zip(pos_idx, pv):
                            if v < VERIFIER["tau"]:
                                masks[j] = np.zeros_like(masks[j]); STATE["vrej"] += 1
                    except Exception:  # noqa: BLE001
                        traceback.print_exc()
                for i, m in zip(ok, masks):
                    ROWS[i] = rle_encode(m)
            except Exception:  # noqa: BLE001
                traceback.print_exc()
                for i in ok:
                    ROWS.setdefault(i, ""); STATE["fail"] += 1
        STATE["done"] += len(ids)
        # 시간 가드: 남은 양을 현재 속도로 환산해 예산을 넘기면 TTA 를 단계적으로 줄인다
        el = time.time() - T_START
        rate = STATE["done"] / max(1e-6, time.time() - T_INFER0)
        eta = (len(IDS) - STATE["done"]) / max(rate, 1e-6)
        if el + eta > TIME_BUDGET_S and STATE["tta"] != "none":
            STATE["tta"] = {"d4": "flip", "flip": "none"}[STATE["tta"]]
            print(f"  [time-guard] 예상 {el + eta:.0f}s > 예산 {TIME_BUDGET_S:.0f}s → TTA={STATE['tta']}")
    n_pos = sum(1 for i in ids_all if ROWS.get(i))
    print(f"shard {k}: {len(ids_all)}건, 양성 {n_pos}건, 2차 기각 누적 {STATE['vrej']}, {time.time() - t0:.1f}s, 누적 {STATE['done']}/{len(IDS)}, 실패 {STATE['fail']}, 마스크 대체 {STATE.get('mask_fallback', 0)}, TTA {STATE['tta']}")


T_INFER0 = time.time()

In [ ]:
# [infer 1/4]
run_shard(0)

In [ ]:
# [infer 2/4]
run_shard(1)

In [ ]:
# [infer 3/4]
run_shard(2)

In [ ]:
# [infer 4/4]
run_shard(3)

In [ ]:
# [save] 누락 id 는 빈 예측으로 채우고, 모든 행을 다시 파싱해 형식 오류를 제출 전에 잡는다
out_rows = []
for i in IDS:
    r = ROWS.get(i, "")
    try:
        n = rle_check(r)
        if 0 < n < MIN_POS_PX:
            r = ""
    except Exception:  # noqa: BLE001
        r = ""
    out_rows.append((i, r))
PREDICTION_PATH.parent.mkdir(parents=True, exist_ok=True)
with PREDICTION_PATH.open("w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "rle"])
    w.writerows(out_rows)
n_pos = sum(1 for _, r in out_rows if r)
print(f"저장: {PREDICTION_PATH} ({PREDICTION_PATH.stat().st_size} bytes), {len(out_rows)}건 중 양성 {n_pos}건 ({n_pos / max(1, len(out_rows)):.1%}), 총 {time.time() - T_START:.1f}s")